# 01 — Almacenamiento en Google Cloud Storage

**Proyecto GCP:** `my-first-project-492901`  
**Bucket:** `big-data-proyecto-parcial`  
**Región:** `us-central1`

Este notebook crea el bucket en GCS y sube los 26 archivos CSV normalizados del dataset Chicago Crimes (2001–2026), organizados bajo el prefijo `raw/`.

```
gs://big-data-proyecto-parcial/
└── raw/
    ├── Chicago_Crimes_2001.csv
    ├── Chicago_Crimes_2002.csv
    ├── ...
    └── Chicago_Crimes_2026.csv
```

### Autenticación
Se usa **Application Default Credentials (ADC)** vía `gcloud auth application-default login` — no se necesita archivo JSON de credenciales.

In [1]:
# !pip install google-cloud-storage

In [2]:
import os
import time
from google.cloud import storage

PROJECT_ID  = 'my-first-project-492901'
BUCKET_NAME = 'big-data-proyecto-parcial'
REGION      = 'us-central1'
LOCAL_FOLDER = 'Chicago_Crimes_by_Year'
GCS_PREFIX   = 'raw'

# ADC: usa las credenciales de 'gcloud auth application-default login'
client = storage.Client(project=PROJECT_ID)
print(f'Cliente GCS inicializado — proyecto: {PROJECT_ID}')

C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Cliente GCS inicializado — proyecto: my-first-project-492901


In [3]:
# ── Crear bucket si no existe ────────────────────────────────────────────────
bucket = client.bucket(BUCKET_NAME)

if not bucket.exists():
    bucket = client.create_bucket(BUCKET_NAME, location=REGION)
    bucket.storage_class = 'STANDARD'
    bucket.patch()
    print(f'Bucket creado:    gs://{BUCKET_NAME}  [{REGION}]')
else:
    print(f'Bucket existente: gs://{BUCKET_NAME}')

print(f'Storage class:    {bucket.storage_class or "STANDARD"}')
print(f'Ubicación:        {bucket.location}')

Bucket existente: gs://big-data-proyecto-parcial
Storage class:    STANDARD
Ubicación:        None


In [4]:
# ── Subir todos los CSVs ─────────────────────────────────────────────────────
csv_files = sorted(f for f in os.listdir(LOCAL_FOLDER) if f.endswith('.csv'))
print(f'Archivos a subir: {len(csv_files)}')
print('─' * 72)

total_bytes = 0
t0 = time.time()

for filename in csv_files:
    local_path = os.path.join(LOCAL_FOLDER, filename)
    gcs_path   = f'{GCS_PREFIX}/{filename}'
    blob       = bucket.blob(gcs_path)
    size_mb    = os.path.getsize(local_path) / (1024 ** 2)
    total_bytes += os.path.getsize(local_path)

    blob.upload_from_filename(local_path, content_type='text/csv')
    print(f'  ✓  {filename:<40}  {size_mb:>7.1f} MB')

elapsed = time.time() - t0
print('─' * 72)
print(f'Total: {total_bytes/(1024**2):.1f} MB en {len(csv_files)} archivos  ({elapsed:.1f}s)')

Archivos a subir: 26
────────────────────────────────────────────────────────────────────────


  ✓  Chicago_Crimes_2001.csv                      94.4 MB


  ✓  Chicago_Crimes_2002.csv                      96.5 MB


  ✓  Chicago_Crimes_2003.csv                      95.8 MB


  ✓  Chicago_Crimes_2004.csv                      94.6 MB


...

  ✓  Chicago_Crimes_2022.csv                      48.7 MB


  ✓  Chicago_Crimes_2023.csv                      53.5 MB


  ✓  Chicago_Crimes_2024.csv                      52.2 MB


  ✓  Chicago_Crimes_2025.csv                      18.9 MB


  ✓  Chicago_Crimes_2026.csv                      10.6 MB
────────────────────────────────────────────────────────────────────────
Total: 1690.2 MB en 26 archivos  (132.8s)


In [5]:
# ── Verificación: listar objetos en el bucket ────────────────────────────────
print(f'Contenido de gs://{BUCKET_NAME}/{GCS_PREFIX}/')
print('─' * 65)

blobs = sorted(client.list_blobs(BUCKET_NAME, prefix=GCS_PREFIX + '/'), key=lambda b: b.name)
total_size = 0
for blob in blobs:
    size_mb = blob.size / (1024 ** 2)
    total_size += blob.size
    print(f'  {blob.name:<52}  {size_mb:>7.1f} MB')

print('─' * 65)
print(f'Total: {len(blobs)} objetos  |  {total_size/(1024**2):.1f} MB')
print()
print('CAPTURA 1 → Consola GCP > Cloud Storage > Buckets (nombre, región, clase)')
print('CAPTURA 2 → Dentro del bucket, carpeta raw/ con los 26 archivos')
print('CAPTURA 3 → Clic en un CSV individual (metadata: tamaño, MIME, URL)')
print('CAPTURA 4 → Pestaña Permissions del bucket')
print('CAPTURA 5 → Esta celda con el output de verificación')

Contenido de gs://big-data-proyecto-parcial/raw/
─────────────────────────────────────────────────────────────────


  raw/Chicago_Crimes_2001.csv                              94.4 MB
  raw/Chicago_Crimes_2002.csv                              96.5 MB
  raw/Chicago_Crimes_2003.csv                              95.8 MB
  raw/Chicago_Crimes_2004.csv                              94.6 MB
  raw/Chicago_Crimes_2005.csv                              91.6 MB
  raw/Chicago_Crimes_2006.csv                              90.6 MB
  raw/Chicago_Crimes_2007.csv                              88.2 MB
  raw/Chicago_Crimes_2008.csv                              86.0 MB
  raw/Chicago_Crimes_2009.csv                              79.2 MB
  raw/Chicago_Crimes_2010.csv                              75.0 MB
  raw/Chicago_Crimes_2011.csv                              71.2 MB
  raw/Chicago_Crimes_2012.csv                              68.0 MB
  raw/Chicago_Crimes_2013.csv                              62.3 MB
  raw/Chicago_Crimes_2014.csv                              56.1 MB
  raw/Chicago_Crimes_2015.csv                              53.

## Justificación del uso de Google Cloud Storage

| Criterio | Justificación |
|---|---|
| **Desacoplamiento cómputo/almacenamiento** | Dask, Spark y BigQuery leen desde GCS sin mover datos — arquitectura data lake estándar de la industria |
| **Durabilidad** | SLA de 99.999999999% (11 nueves) con replicación geográfica automática |
| **Escalabilidad** | El dataset actual (~3 GB) escala a TB sin cambiar arquitectura ni reprovisionar |
| **Integración nativa GCP** | BigQuery External Tables y Dataproc leen GCS sin ETL adicional |
| **Costo optimizado** | Datos históricos pre-2020 elegibles para clase Nearline (40% más barato) |
| **Partition pruning** | 26 archivos por año permiten leer solo los años necesarios en cada consulta |

---
## Clúster Dataproc — Procesamiento Distribuido en GCP

Google Cloud Dataproc es el servicio administrado de GCP para ejecutar Apache Spark, Hadoop y otros frameworks de Big Data sobre clústeres efímeros. Permite escalar el mismo código PySpark que corre localmente (en `04_spark_crud.ipynb`) a un clúster real sin modificar una línea.

### Justificación Técnica del Dimensionamiento

El clúster se diseña en base a las características reales del dataset:

| Parámetro | Valor | Razonamiento |
|---|---|---|
| **Dataset** | 26 CSVs · 1.94 GB | Un clúster pequeño es suficiente para este volumen |
| **Nodo Master** | 1 × `n4-standard-4` (4 vCPU, 16 GB RAM) | Coordina el DAG y ejecuta el Driver de Spark |
| **Workers** | 2 × `n4-standard-4` (4 vCPU, 16 GB RAM c/u) | Mínimo recomendado por GCP; procesa ~1 GB/worker |
| **Disco por nodo** | 100 GB `hyperdisk-balanced` | Dataset cabe 3× en disco con espacio para shuffle |
| **Región** | `us-central1` | **Misma región que el bucket GCS** → transferencia gratuita y baja latencia |
| **Versión imagen** | `2.3-debian12` | Spark 3.5 + Python 3.11 — compatible con nuestro código |
| **Componentes** | Jupyter | Permite ejecutar notebooks directamente en el clúster |

**¿Por qué 2 workers y no más?**  
Para ~2 GB de datos, el bottleneck es la lectura de GCS, no el cómputo. Agregar más workers solo incrementa el costo sin reducir el tiempo significativamente. En un escenario real con los 26 años completos (26+ GB o más), se escalaría a 4–8 workers `n4-standard-8`.

**¿Por qué `us-central1`?**  
El bucket `big-data-proyecto-parcial` está en `us-central1`. La transferencia de datos entre GCS y Dataproc en la **misma región** es gratuita y opera sobre la red interna de Google (no internet público), eliminando el principal costo oculto de los clústeres de Big Data.

In [6]:
from google.cloud import dataproc_v1
import time

# ── Configuración del clúster ─────────────────────────────────────────────────
PROJECT_ID   = 'my-first-project-492901'
REGION       = 'us-central1'
ZONE         = 'us-central1-b'
CLUSTER_NAME = 'chicago-crimes'       # Clúster exclusivo para este proyecto
BUCKET_NAME  = 'big-data-proyecto-parcial'

CLUSTER_CONFIG = {
    "cluster_name": CLUSTER_NAME,
    "project_id":   PROJECT_ID,
    "config": {
        "config_bucket": BUCKET_NAME,
        "gce_cluster_config": {
            "zone_uri": f"projects/{PROJECT_ID}/zones/{ZONE}",
            "network_uri": "default",
            "internal_ip_only": False,
        },
        "master_config": {
            "num_instances": 1,
            "machine_type_uri": "n4-standard-4",
            "disk_config": {"boot_disk_type": "hyperdisk-balanced", "boot_disk_size_gb": 100},
        },
        "worker_config": {
            "num_instances": 2,
            "machine_type_uri": "n4-standard-4",
            "disk_config": {"boot_disk_type": "hyperdisk-balanced", "boot_disk_size_gb": 100},
        },
        "software_config": {
            "image_version": "2.3-debian12",
            "optional_components": [dataproc_v1.Component.JUPYTER],
            "properties": {
                "spark:spark.executor.memory":        "10g",
                "spark:spark.driver.memory":          "6g",
                "spark:spark.sql.shuffle.partitions": "16",
                "spark:spark.executor.instances":     "2",
            },
        },
    },
}

print('═' * 60)
print(f'  Clúster : {CLUSTER_NAME}')
print(f'  Proyecto: {PROJECT_ID}')
print(f'  Región  : {REGION}  /  Zona: {ZONE}')
print('─' * 60)
print(f'  Master  : 1 × n4-standard-4  (4 vCPU, 16 GB, 100 GB hyperdisk)')
print(f'  Workers : 2 × n4-standard-4  (4 vCPU, 16 GB, 100 GB hyperdisk)')
print(f'  Imagen  : 2.3-debian12  →  Spark 3.5 + Python 3.11')
print(f'  Bucket  : gs://{BUCKET_NAME}  (misma región → red interna gratis)')
print('═' * 60)

════════════════════════════════════════════════════════════
  Clúster : chicago-crimes
  Proyecto: my-first-project-492901
  Región  : us-central1  /  Zona: us-central1-b
────────────────────────────────────────────────────────────
  Master  : 1 × n4-standard-4  (4 vCPU, 16 GB, 100 GB hyperdisk)
  Workers : 2 × n4-standard-4  (4 vCPU, 16 GB, 100 GB hyperdisk)
  Imagen  : 2.3-debian12  →  Spark 3.5 + Python 3.11
  Bucket  : gs://big-data-proyecto-parcial  (misma región → red interna gratis)
════════════════════════════════════════════════════════════


In [7]:
# ── Verificar clúster Dataproc chicago-crimes ─────────────────────────────────
cluster_client = dataproc_v1.ClusterControllerClient(
    client_options={"api_endpoint": f"{REGION}-dataproc.googleapis.com:443"}
)

result = cluster_client.get_cluster(
    request={"project_id": PROJECT_ID, "region": REGION, "cluster_name": CLUSTER_NAME}
)

master_names = list(result.config.master_config.instance_names)
worker_names = list(result.config.worker_config.instance_names)

print('═' * 60)
print(f'  Clúster : {result.cluster_name}')
print(f'  Estado  : {result.status.state.name}')
print('─' * 60)
print(f'  Master  : {master_names[0] if master_names else "(pendiente)"}')
print(f'  Workers : {worker_names}')
print(f'  Imagen  : {result.config.software_config.image_version}')
print(f'  Bucket  : gs://{result.config.config_bucket}')
print(f'  Región  : {REGION}')
print('═' * 60)
print()
print('✓ Clúster chicago-crimes — exclusivo para el proyecto Big Data Chicago Crimes')
print('CAPTURA → Consola GCP > Dataproc > Clusters: mostrar estado RUNNING')

C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


════════════════════════════════════════════════════════════
  Clúster : chicago-crimes
  Estado  : RUNNING
────────────────────────────────────────────────────────────
  Master  : chicago-crimes-m
  Workers : ['chicago-crimes-w-0', 'chicago-crimes-w-1']
  Imagen  : 2.3.29-debian12
  Bucket  : gs://big-data-proyecto-parcial
  Región  : us-central1
════════════════════════════════════════════════════════════

✓ Clúster chicago-crimes — exclusivo para el proyecto Big Data Chicago Crimes
CAPTURA → Consola GCP > Dataproc > Clusters: mostrar estado RUNNING


In [8]:
# ── Confirmar acceso del clúster al bucket GCS ────────────────────────────────
print(f'Verificando acceso al bucket gs://{BUCKET_NAME}/raw/ ...\n')

gcs_client = storage.Client(project=PROJECT_ID)
blobs = list(gcs_client.list_blobs(BUCKET_NAME, prefix='raw/Chicago_Crimes_'))
csv_blobs = [b for b in blobs if b.name.endswith('.csv')]

print(f'  Archivos CSV visibles desde el clúster (vía GCS nativo): {len(csv_blobs)}')
for b in csv_blobs[:5]:
    print(f'    gs://{BUCKET_NAME}/{b.name}  ({b.size/(1024**2):.1f} MB)')
print(f'    ...')
total_gb = sum(b.size for b in csv_blobs) / (1024**3)
print(f'  Total: {total_gb:.2f} GB en {len(csv_blobs)} archivos')
print()
print('Comandos para conectar al clúster manualmente:')
print(f'  gcloud compute ssh {CLUSTER_NAME}-m --zone={ZONE} \\')
print(f'    --command="gsutil ls -l gs://{BUCKET_NAME}/raw/ | tail -3"')
print()
print('✓ El clúster lee directamente gs://big-data-proyecto-parcial/raw/*.csv')
print('  Sin copiar datos — Spark usa el conector GCS nativo (gs:// URI scheme)')

Verificando acceso al bucket gs://big-data-proyecto-parcial/raw/ ...



C:\Users\Usuario\miniconda3\envs\bigdata\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


  Archivos CSV visibles desde el clúster (vía GCS nativo): 26
    gs://big-data-proyecto-parcial/raw/Chicago_Crimes_2001.csv  (94.4 MB)
    gs://big-data-proyecto-parcial/raw/Chicago_Crimes_2002.csv  (96.5 MB)
    gs://big-data-proyecto-parcial/raw/Chicago_Crimes_2003.csv  (95.8 MB)
    gs://big-data-proyecto-parcial/raw/Chicago_Crimes_2004.csv  (94.6 MB)
    gs://big-data-proyecto-parcial/raw/Chicago_Crimes_2005.csv  (91.6 MB)
    ...
  Total: 1.65 GB en 26 archivos

Comandos para conectar al clúster manualmente:
  gcloud compute ssh chicago-crimes-m --zone=us-central1-b \
    --command="gsutil ls -l gs://big-data-proyecto-parcial/raw/ | tail -3"

✓ El clúster lee directamente gs://big-data-proyecto-parcial/raw/*.csv
  Sin copiar datos — Spark usa el conector GCS nativo (gs:// URI scheme)


### Cómo escalar el código local a Dataproc

El mismo script PySpark de `04_spark_crud.ipynb` se ejecuta en Dataproc cambiando únicamente la ruta de los archivos:

```python
# LOCAL — lee desde disco
files = [f'Chicago_Crimes_by_Year/Chicago_Crimes_{y}.csv' for y in range(2017, 2026)]

# DATAPROC — exactamente el mismo código, solo cambia el prefijo
files = [f'gs://big-data-proyecto-parcial/raw/Chicago_Crimes_{y}.csv' for y in range(2017, 2026)]
```

### Eliminación del clúster (buenas prácticas de costos)

```python
# Eliminar clúster al finalizar (ejecutar manualmente)
operation = cluster_client.delete_cluster(
    request={"project_id": PROJECT_ID, "region": REGION, "cluster_name": CLUSTER_NAME}
)
operation.result()
print(f'Clúster {CLUSTER_NAME} eliminado.')
```

> **Nota:** Los datos en GCS permanecen intactos tras eliminar el clúster. El almacenamiento y el cómputo son independientes — principio fundamental de la arquitectura GCP.